# Step 6 — Giving the Agents Memory

An LLM call remembers **nothing**. A follow-up like *"Can it be split into two parts?"* only makes sense if the earlier conversation is sent again. We try the same two-turn chat three ways:

- **A.** No memory → the follow-up loses its topic
- **B.** Memory **by hand** → we pass the earlier turns in ourselves (Step 3 agent)
- **C.** **Framework** memory → LangGraph's checkpointer stores the chat under a `thread_id` (Step 4 agent)

### Setup
Run this cell first. It lets the notebook import the lab's own code (`config.py`, `rag/`, `agent/`, `llm.py`) from the folder above `notebooks/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("Project folder:", ROOT)

In [ ]:
from rag.ingest import build_index

build_index()

In [ ]:
from llm import check_llm

ok, message = check_llm()
print("LLM ready ✅" if ok else f"LLM NOT ready ❌  {message}")

In [ ]:
from agent import print_trace
from agent.graph_agent import build_graph, model_view, run_graph_agent
from agent.react_agent import run_react_agent

FIRST = "How long is Paternity Leave?"
FOLLOW_UP = "Can it be split into two parts?"   # "it" = Paternity Leave — only memory can tell

## A. No memory

In [ ]:
print_trace(run_react_agent(FIRST))
print("-" * 80)
print_trace(run_react_agent(FOLLOW_UP))

👉 Look at the second search: it has no idea what "it" is.

## B. Memory by hand
The memory is just a Python list. We add each question and its final answer, and send the list with the next call.

In [ ]:
history = []
for question in (FIRST, FOLLOW_UP):
    result = run_react_agent(question, history=history)
    print(f"Q: {question}   (agent saw {result['memory_messages']} earlier messages)")
    print_trace(result)
    print("-" * 80)
    history += [{"role": "user", "content": question}, {"role": "assistant", "content": result["answer"]}]

history

## C. Framework memory (LangGraph checkpointer)
Same `thread_id` → the conversation continues. The checkpointer stores *everything*; `model_view()` chooses what the model sees.

In [ ]:
THREAD = "notebook-demo"
for question in (FIRST, FOLLOW_UP):
    result = run_graph_agent(question, thread_id=THREAD)
    print(f"Q: {question}   (agent saw {result['memory_messages']} earlier messages)")
    print_trace(result)
    print("-" * 80)

In [ ]:
stored = build_graph().get_state({"configurable": {"thread_id": THREAD}}).values["messages"]
visible, _ = model_view(stored)
print(f"Stored by the checkpointer: {len(stored)} messages")
print(f"Shown to the model:         {len(visible)} messages")

A **new** `thread_id` is a new conversation: nothing carries over.

In [ ]:
print_trace(run_graph_agent(FOLLOW_UP, thread_id="another-chat"))

## What memory costs
- Every earlier message is sent again on **every** call: more tokens, slower answers, and old turns can distract a small model. That's why only the last `MEMORY_MAX_MESSAGES` (10) are kept.
- This memory lives in RAM. Restarting the kernel forgets it.